In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import warnings
import tensorflow as tf
import pandas as pd

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import run_unet_mlp_uv15 as base
import run_unet_mlp_bottleneckvqc_uv15_randomsplit as exp

from functions.nb_helpers import (
    signed_log1p,
    transform_y_signed_log1p,
    repo_root,
    resolve_extracted_uv_dir,
    evaluate_split_physical,
    component_energy_spectrum,
    make_band_table,
    plot_spectral_error,
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
height_m = 15.0
last_k = 12
epochs = 1000
batch_size = 2
cond_emb_dim = 16
n_qubits = 5
n_layers = 2
seed = 7
train_frac = 0.8
val_frac = 0.1

base.set_seeds(seed)


In [ ]:
here = str(repo_root())
os.chdir(here)

extracted_uv_dir = str(resolve_extracted_uv_dir(here))

cases = base.list_cases(extracted_uv_dir)

xs, cs, ys, meta = [], [], [], []
for c in cases:
    m_building, u_mean, v_mean = base.load_uv_steady_mean(c.path, height_m=height_m, last_k=last_k)
    x = m_building[..., None].astype(np.float32)
    y = np.stack([u_mean, v_mean], axis=-1).astype(np.float32)
    xs.append(x)
    cs.append(base.cond_vector(c.speed, c.angle_deg))
    ys.append(y)
    meta.append({"file": os.path.basename(c.path), "speed": c.speed, "d_code": c.d_code, "angle_deg": c.angle_deg})

X = np.stack(xs, axis=0)
C = np.stack(cs, axis=0)
Y = np.stack(ys, axis=0)

X.shape, C.shape, Y.shape


In [ ]:
# Non-building u_mean / v_mean distributions
u_all = Y[..., 0]
v_all = Y[..., 1]

valid_mask = (u_all != base.MISSING_VALUE) & (v_all != base.MISSING_VALUE)
u_valid = u_all[valid_mask]
v_valid = v_all[valid_mask]

# Signed log1p transform: x -> sign(x) * log1p(|x|)

u_log = signed_log1p(u_valid)
v_log = signed_log1p(v_valid)

# Train targets in signed-log1p space (buildings keep missing value)
Y_log = transform_y_signed_log1p(Y, base.MISSING_VALUE)


In [ ]:
# random split
if train_frac + val_frac >= 1.0:
    raise ValueError(f"train_frac+val_frac must be < 1. Got {train_frac+val_frac}")

split_idx = exp._random_split_indices(n=len(cases), train_frac=train_frac, val_frac=val_frac, seed=seed)
tr, va, te = split_idx["train"], split_idx["val"], split_idx["test"]

# Condition scaling (fit on train only)
c_scaler = base.StandardScaler()
C_tr = c_scaler.fit_transform(C[tr])
C_va = c_scaler.transform(C[va])
C_te = c_scaler.transform(C[te])

# Output normalization in signed-log1p space (fit on train only, ignoring missing)
y_mean, y_std = base.compute_y_norm_stats(Y_log[tr])
Y_tr = base.normalize_y(Y_log[tr], y_mean, y_std)
Y_va = base.normalize_y(Y_log[va], y_mean, y_std)
Y_te = base.normalize_y(Y_log[te], y_mean, y_std)

X_tr, X_va, X_te = X[tr], X[va], X[te]
len(tr), len(va), len(te)


# C-QB-UNet

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_5q2l.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model = exp.build_unet_cond_mlp_bottleneck_vqc(
    input_shape=X_tr.shape[1:],
    cond_dim=C_tr.shape[-1],
    cond_emb_dim=cond_emb_dim,
    n_qubits=n_qubits,
    n_layers=n_layers,
)

model.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model.load_weights(best_weights_path)

print("Train/Val/Test metrics (evaluated in m/s after inverse transform):")
Y_tr_denorm, pred_tr_denorm, metrics_tr = evaluate_split_physical(model, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm, pred_va_denorm, metrics_va = evaluate_split_physical(model, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm, pred_te_denorm, metrics_te = evaluate_split_physical(model, "test", X_te, C_te, Y_te, y_mean, y_std)

# C-UNet

In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])

model_mlp.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model_mlp.load_weights(best_weights_path)

print("C-UNet Train/Val/Test metrics (evaluated in m/s after inverse transform):")
Y_tr_denorm_mlp, pred_tr_denorm_mlp, metrics_tr_mlp = evaluate_split_physical(model_mlp, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm_mlp, pred_va_denorm_mlp, metrics_va_mlp = evaluate_split_physical(model_mlp, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm_mlp, pred_te_denorm_mlp, metrics_te_mlp = evaluate_split_physical(model_mlp, "test", X_te, C_te, Y_te, y_mean, y_std)

# 2D FFT Spatial Spectrum

In [ ]:
# ===== 1) Build ROI mask (top-left open area) =====
tl_h, tl_w = 75, 90
H, W = Y_te_denorm.shape[1], Y_te_denorm.shape[2]
roi2d = np.zeros((H, W), dtype=bool)
roi2d[H - tl_h : H, :tl_w] = True  # consistent with earlier visualization

# Non-building valid region (from GT)
valid_all = (Y_te_denorm[..., 0] != base.MISSING_VALUE)   # [N,H,W]

# Keep valid pixels inside ROI only
valid_roi = valid_all & roi2d[None, :]

# ===== 2) Reuse component_energy_spectrum =====
dx, dy = 1.0, 1.0
n_bins = 80
eps = 1e-20

# u
k, E_u_gt  = component_energy_spectrum(Y_te_denorm,        valid_roi, comp=0, dx=dx, dy=dy, n_bins=n_bins)
_, E_u_mlp = component_energy_spectrum(pred_te_denorm_mlp, valid_roi, comp=0, dx=dx, dy=dy, n_bins=n_bins)
_, E_u_vqc = component_energy_spectrum(pred_te_denorm,     valid_roi, comp=0, dx=dx, dy=dy, n_bins=n_bins)

# v
_, E_v_gt  = component_energy_spectrum(Y_te_denorm,        valid_roi, comp=1, dx=dx, dy=dy, n_bins=n_bins)
_, E_v_mlp = component_energy_spectrum(pred_te_denorm_mlp, valid_roi, comp=1, dx=dx, dy=dy, n_bins=n_bins)
_, E_v_vqc = component_energy_spectrum(pred_te_denorm,     valid_roi, comp=1, dx=dx, dy=dy, n_bins=n_bins)

# total = u + v
E_t_gt  = E_u_gt  + E_v_gt
E_t_mlp = E_u_mlp + E_v_mlp
E_t_vqc = E_u_vqc + E_v_vqc

# ===== 3) Plot: 1x3 =====
fig, axes = plt.subplots(1, 3, figsize=(9, 4))

axes[0].loglog(k, E_u_gt  + eps, label="Truth", lw=2)
axes[0].loglog(k, E_u_mlp + eps, label="C-UNet", lw=2)
axes[0].loglog(k, E_u_vqc + eps, label="C-QB-UNet", lw=2)
axes[0].set_title("U Spectrum (Open Area)")
axes[0].set_xlabel("Wavenumber k (rad/m)")
axes[0].set_ylabel("E_u(k)")
axes[0].grid(False)
axes[0].legend(frameon=False)

axes[1].loglog(k, E_v_gt  + eps, label="Truth", lw=2)
axes[1].loglog(k, E_v_mlp + eps, label="C-UNet", lw=2)
axes[1].loglog(k, E_v_vqc + eps, label="C-QB-UNet", lw=2)
axes[1].set_title("V Spectrum (Open Area)")
axes[1].set_xlabel("Wavenumber k (rad/m)")
axes[1].set_ylabel("E_v(k)")
axes[1].grid(False)
axes[1].legend(frameon=False)

axes[2].loglog(k, E_t_gt  + eps, label="Truth", lw=2)
axes[2].loglog(k, E_t_mlp + eps, label="C-UNet", lw=2)
axes[2].loglog(k, E_t_vqc + eps, label="C-QB-UNet", lw=2)
axes[2].set_title("Total Spectrum (U+V, Open Area)")
axes[2].set_xlabel("Wavenumber k (rad/m)")
axes[2].set_ylabel("E_total(k)")
axes[2].grid(False)
axes[2].legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# ===== 1) Build mask outside the top-left ROI =====
# Requires earlier variables:
# roi2d, valid_all, component_energy_spectrum(...)

valid_rest = valid_all & (~roi2d[None, :])  # [N,H,W]

# ===== 2) Reuse spectrum helper for u/v/total =====
dx, dy = 1.0, 1.0
n_bins = 80
eps = 1e-20

# u
k, E_u_gt_r  = component_energy_spectrum(Y_te_denorm,        valid_rest, comp=0, dx=dx, dy=dy, n_bins=n_bins)
_, E_u_mlp_r = component_energy_spectrum(pred_te_denorm_mlp, valid_rest, comp=0, dx=dx, dy=dy, n_bins=n_bins)
_, E_u_vqc_r = component_energy_spectrum(pred_te_denorm,     valid_rest, comp=0, dx=dx, dy=dy, n_bins=n_bins)

# v
_, E_v_gt_r  = component_energy_spectrum(Y_te_denorm,        valid_rest, comp=1, dx=dx, dy=dy, n_bins=n_bins)
_, E_v_mlp_r = component_energy_spectrum(pred_te_denorm_mlp, valid_rest, comp=1, dx=dx, dy=dy, n_bins=n_bins)
_, E_v_vqc_r = component_energy_spectrum(pred_te_denorm,     valid_rest, comp=1, dx=dx, dy=dy, n_bins=n_bins)

# total
E_t_gt_r  = E_u_gt_r  + E_v_gt_r
E_t_mlp_r = E_u_mlp_r + E_v_mlp_r
E_t_vqc_r = E_u_vqc_r + E_v_vqc_r

# ===== 3) Plot =====
fig, axes = plt.subplots(1, 3, figsize=(9, 4))

axes[0].loglog(k, E_u_gt_r  + eps, label="Truth", lw=2)
axes[0].loglog(k, E_u_mlp_r + eps, label="C-UNet", lw=2)
axes[0].loglog(k, E_u_vqc_r + eps, label="C-QB-UNet", lw=2)
axes[0].set_title("U Spectrum (Built-up Area)")
axes[0].set_xlabel("Wavenumber k (rad/m)")
axes[0].set_ylabel("E_u(k)")
axes[0].grid(False)
axes[0].legend(frameon=False)

axes[1].loglog(k, E_v_gt_r  + eps, label="Truth", lw=2)
axes[1].loglog(k, E_v_mlp_r + eps, label="C-UNet", lw=2)
axes[1].loglog(k, E_v_vqc_r + eps, label="C-QB-UNet", lw=2)
axes[1].set_title("V Spectrum (Built-up Area)")
axes[1].set_xlabel("Wavenumber k (rad/m)")
axes[1].set_ylabel("E_v(k)")
axes[1].grid(False)
axes[1].legend(frameon=False)

axes[2].loglog(k, E_t_gt_r  + eps, label="Truth", lw=2)
axes[2].loglog(k, E_t_mlp_r + eps, label="C-UNet", lw=2)
axes[2].loglog(k, E_t_vqc_r + eps, label="C-QB-UNet", lw=2)
axes[2].set_title("Total Spectrum (U+V, Built-up Area)")
axes[2].set_xlabel("Wavenumber k (rad/m)")
axes[2].set_ylabel("E_total(k)")
axes[2].grid(False)
axes[2].legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
#Band-energy ratios and spectral-error curves
eps = 1e-20

# Helpers

# -----------------------------
# 1) Define band intervals (example)
# Adjust cutoffs to match the physics of interest
# -----------------------------
k_min, k_max = float(np.min(k)), float(np.max(k))
k1 = k_min + 0.33 * (k_max - k_min)
k2 = k_min + 0.66 * (k_max - k_min)

band_defs = [
    ("low",  k_min, k1),
    ("mid",  k1,    k2),
    ("high", k2,    k_max + 1e-12),
]

print("Band definitions:")
for b in band_defs:
    print(b)

# -----------------------------
# 2) Top-left ROI: total spectrum
# Requires: k, E_t_gt, E_t_mlp, E_t_vqc
# -----------------------------
df_top_total = make_band_table(
    k, E_t_gt, E_t_mlp, E_t_vqc, band_defs, region_name="Open Area (total)"
)
display(df_top_total)

plot_spectral_error(
    k, E_t_gt, E_t_mlp, E_t_vqc, title="Open Area"
)

# -----------------------------
# 3) Outside ROI: total spectrum
# Requires: k, E_t_gt_r, E_t_mlp_r, E_t_vqc_r
# Reuse the same k if outside-ROI spectra share bins
# -----------------------------
df_out_total = make_band_table(
    k, E_t_gt_r, E_t_mlp_r, E_t_vqc_r, band_defs, region_name="Built-up Area"
)
display(df_out_total)

plot_spectral_error(
    k, E_t_gt_r, E_t_mlp_r, E_t_vqc_r, title="Built-up Area"
)

# -----------------------------
# 4) Concatenate into one table (optional save)
# -----------------------------
df_all = pd.concat([df_top_total, df_out_total], ignore_index=True)
display(df_all)
